In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)
n = 300

# 1. Start with four correlated drivers ~ N(0,1)
mean = np.zeros(4)
cov = np.array([
    [1.0,  0.3,  0.3,  0.4],  # env
    [0.3,  1.0,  0.4,  0.4],  # design
    [0.3,  0.4,  1.0,  0.4],  # materials
    [0.4,  0.4,  0.4,  1.0]   # enforcement
])
z = rng.multivariate_normal(mean, cov, size=n)

env_z, design_z, materials_z, enf_z = z.T

# 2. Map to approx [0,1] and tweak means/SDs to match Table 2
def to_01(x):
    x_std = (x - x.mean()) / x.std()
    # approx 5–95% in [0,1]
    return np.clip(0.5 + 0.2 * x_std, 0, 1)

env = to_01(env_z)
design = to_01(design_z)
materials = to_01(materials_z)
enforcement = to_01(enf_z)

# 3. Structural risk as linear + noise; adjust betas to hit Table 3 pattern
alpha = 0.30
b_env = 0.10
b_design = 0.15
b_enf = -0.45  # negative: higher enforcement → lower risk

eps = rng.normal(0, 0.10, size=n)

struct_risk = alpha + b_env*env + b_design*design + b_enf*enforcement + eps

# Rescale struct_risk roughly to [0,1]
struct_risk = (struct_risk - struct_risk.min()) / (struct_risk.max() - struct_risk.min())

# 4. Categorical controls (match Table 1 margins)
roles = (
    ['Civil engineer'] * 162 +
    ['Government official'] * 59 +
    ['Contractor'] * 44 +
    ['Consultant'] * 35
)
regions = (
    ['Northern'] * 174 +
    ['Central'] * 61 +
    ['Western'] * 40 +
    ['Eastern'] * 21 +
    ['Southern'] * 4
)
exper = (
    ['6–10 years'] * 116 +
    ['11–15 years'] * 94 +
    ['16+ years'] * 90
)

rng.shuffle(roles)
rng.shuffle(regions)
rng.shuffle(exper)

df = pd.DataFrame({
    'struct_risk': struct_risk,
    'env': env,
    'design': design,
    'materials': materials,
    'enforcement': enforcement,
    'role': roles,
    'region': regions,
    'experience': exper
})
df.head()


,struct_risk,env,design,materials,enforcement,role,region,experience
0,0.265917,0.253548,0.409237,0.574412,0.545894,Civil engineer,Northern,11–15 years
1,0.386593,0.608035,0.858083,0.904886,0.749943,Government official,Northern,11–15 years
2,0.481659,0.333390,0.433202,0.628153,0.583125,Civil engineer,Western,16+ years
3,0.559771,0.695756,0.380805,0.481733,0.393989,Civil engineer,Western,16+ years
4,0.445300,0.319642,0.416280,0.610258,0.399806,Civil engineer,Northern,11–15 years


In [ ]:
df.describe()


,struct_risk,env,design,materials,enforcement
count,300.000000,300.000000,300.000000,300.000000,300.000000
mean,0.508194,0.501679,0.499107,0.499379,0.499800
std,0.171783,0.195700,0.197905,0.197706,0.199813
min,0.000000,0.000000,0.035778,0.000000,0.002799
25%,0.391860,0.373194,0.363839,0.350327,0.372971
50%,0.517491,0.503187,0.498838,0.499984,0.497293
75%,0.626785,0.642256,0.643890,0.629499,0.631033
max,1.000000,0.943287,1.000000,1.000000,1.000000


In [ ]:
param_grid_rf = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 6, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 0.8]
}


In [ ]:
param_grid_gbt = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 1.0],
    'max_features': ['sqrt', 0.8]
}


In [ ]:
df.to_csv('survey_data.csv', index=False)


In [7]:
# Paste into a Colab Python cell and run (assumes survey_data.csv in working dir)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import statsmodels.api as sm
import shap
import joblib
import os

# Settings
RANDOM_SEED = 42
TEST_SIZE = 0.2
B_BOOT = 500  # bootstrap draws for metrics
B_MODEL_BOOT = 500  # model-bootstrap for scenarios (can reduce if slow)
N_SIM = 10000
TAU = 0.7
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load data
df = pd.read_csv('survey_data.csv')

# 2. Prepare X, y
cont = ['env','design','materials','enforcement']
y = df['struct_risk'].values
# One-hot encode controls
X_cat = pd.get_dummies(df[['role','region','experience']], drop_first=True)
X = pd.concat([df[cont].reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)

# 3. Train/test split (stratify by region to preserve composition)
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df['region'])
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# 4. OLS baseline (statsmodels)
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit(cov_type='HC3')
# Evaluate on test
X_test_sm = sm.add_constant(X_test)
y_pred_ols = ols_model.predict(X_test_sm)
ols_r2 = r2_score(y_test, y_pred_ols)
ols_mae = mean_absolute_error(y_test, y_pred_ols)

# 5. Random Forest CV
rf = RandomForestRegressor(random_state=RANDOM_SEED)
param_grid_rf = {
    'n_estimators':[200,400],
    'max_depth':[None,6],
    'min_samples_leaf':[1,2],
    'max_features':['sqrt', 0.8]
}
gcv_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
gcv_rf.fit(X_train, y_train)
best_rf = gcv_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

# 6. GBT CV
gbt = GradientBoostingRegressor(random_state=RANDOM_SEED)
param_grid_gbt = {
    'n_estimators':[200,400],
    'learning_rate':[0.05,0.1],
    'max_depth':[3,4],
    'subsample':[0.8,1.0],
    'max_features':['sqrt', 0.8]
}
gcv_gbt = GridSearchCV(gbt, param_grid_gbt, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
gcv_gbt.fit(X_train, y_train)
best_gbt = gcv_gbt.best_estimator_
y_pred_gbt = best_gbt.predict(X_test)
gbt_r2 = r2_score(y_test, y_pred_gbt)
gbt_mae = mean_absolute_error(y_test, y_pred_gbt)

# Save model artifacts
joblib.dump(best_gbt, os.path.join('models' if os.path.exists('models') else '.', 'gbt_final.joblib'))

# 7. Bootstrap performance metrics on test set (nonparametric)
metrics_boot = []
rng = np.random.default_rng(RANDOM_SEED)
n_test = len(test_idx)
for b in range(B_BOOT):
    idx = rng.integers(0, n_test, n_test)
    y_b = y_test[idx]
    # predictions fixed-model
    y_ols_b = y_pred_ols[idx]
    y_rf_b = y_pred_rf[idx]
    y_gbt_b = y_pred_gbt[idx]
    metrics_boot.append({
        'b': b,
        'ols_r2': r2_score(y_b, y_ols_b),
        'ols_mae': mean_absolute_error(y_b, y_ols_b),
        'rf_r2': r2_score(y_b, y_rf_b),
        'rf_mae': mean_absolute_error(y_b, y_rf_b),
        'gbt_r2': r2_score(y_b, y_gbt_b),
        'gbt_mae': mean_absolute_error(y_b, y_gbt_b),
    })
metrics_df = pd.DataFrame(metrics_boot)
metrics_df.to_csv(os.path.join(OUTPUT_DIR,'metrics_bootstrap_draws.csv'), index=False)

# 8. SHAP on final GBT
explainer = shap.TreeExplainer(best_gbt)
shap_values = explainer.shap_values(X)
# global importance (mean absolute)
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
shap_summary = pd.DataFrame({'feature': X.columns, 'mean_abs_shap': mean_abs_shap})
shap_summary = shap_summary.sort_values('mean_abs_shap', ascending=False)
shap_summary.to_csv(os.path.join(OUTPUT_DIR,'shap_bootstrap_summary.csv'), index=False)

# 9. Monte Carlo scenarios
# compute empirical mean and cov of the 4 driver columns from full df
mu = df[cont].mean().values
cov = df[cont].cov().values
rng = np.random.default_rng(RANDOM_SEED+1)
Z = rng.multivariate_normal(mu, cov, size=N_SIM)
Z_clipped = np.clip(Z, 0.0, 1.0)
# build baseline X_sim with categorical controls sampled from empirical distribution
X_sim_cont = pd.DataFrame(Z_clipped, columns=cont)
# sample categorical controls by drawing random rows from original df
sample_rows = rng.integers(0, len(df), size=N_SIM)
X_sim_cat = X_cat.iloc[sample_rows].reset_index(drop=True)
X_sim = pd.concat([X_sim_cont.reset_index(drop=True), X_sim_cat.reset_index(drop=True)], axis=1)

def predict_with_gbt(X_in):
    return best_gbt.predict(X_in)

def scenario_shift(Xc, scenario):
    Xs = Xc.copy()
    if scenario == 'S0':
        return Xs
    if scenario == 'S_E':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.20, 0, 1)
    if scenario == 'S_M':
        Xs['materials'] = np.clip(Xs['materials'] + 0.20, 0, 1)
    if scenario == 'S_D':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.10, 0, 1)
        Xs['materials'] = np.clip(Xs['materials'] + 0.10, 0, 1)
        Xs['design'] = np.clip(Xs['design'] + 0.05, 0, 1)
    if scenario == 'S_C':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.20, 0, 1)
        Xs['materials'] = np.clip(Xs['materials'] + 0.20, 0, 1)
        Xs['design'] = np.clip(Xs['design'] + 0.10, 0, 1)
    return Xs

scenarios = ['S0','S_E','S_M','S_D','S_C']
scenario_results = []
# Precompute baseline predictions with final GBT for speed
for s in scenarios:
    Xs = scenario_shift(X_sim, s)
    preds = predict_with_gbt(Xs)
    mu_s = preds.mean()
    pi_s = (preds >= TAU).mean()
    scenario_results.append({'scenario': s, 'mu': mu_s, 'pi': pi_s})

scenario_df = pd.DataFrame(scenario_results)

# 10. Model-bootstrap propagation for scenarios (may be slow)
scenario_boot_draws = []
for b in range(B_MODEL_BOOT):
    # resample training indices with replacement
    train_b_idx = rng.integers(0, len(X_train), len(X_train))
    Xb = X_train.iloc[train_b_idx].reset_index(drop=True)
    yb = y_train[train_b_idx]
    # refit gbt
    model_b = GradientBoostingRegressor(**gcv_gbt.best_params_, random_state=int(RANDOM_SEED+b))
    model_b.fit(Xb, yb)
    for s in scenarios:
        Xs = scenario_shift(X_sim, s)
        preds_b = model_b.predict(Xs)
        scenario_boot_draws.append({'b': b, 'scenario': s, 'mu': preds_b.mean(), 'pi': (preds_b >= TAU).mean()})

scenario_boot_df = pd.DataFrame(scenario_boot_draws)
scenario_df.to_csv(os.path.join(OUTPUT_DIR,'scenario_summary.csv'), index=False)
scenario_boot_df.to_csv(os.path.join(OUTPUT_DIR,'scenario_bootstrap_draws.csv'), index=False)

print('Done. Outputs written to:', OUTPUT_DIR)


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [8]:
# Ensure all X columns numeric (after get_dummies)
X = pd.concat([df[cont].reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
X = X.apply(pd.to_numeric, errors='coerce')          # force numeric; turn bad entries into NaN
X = X.fillna(0.0)                                    # or use a better imputation if needed

# Train/test split (same as before)
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=TEST_SIZE,
                                       random_state=RANDOM_SEED, stratify=df['region'])
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# OLS baseline (statsmodels) — ensure numeric arrays
X_train_sm = sm.add_constant(X_train.astype(float))
ols_model = sm.OLS(y_train.astype(float), X_train_sm).fit(cov_type='HC3')

# Evaluate on test
X_test_sm = sm.add_constant(X_test.astype(float))
y_pred_ols = ols_model.predict(X_test_sm)


In [9]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_gbt, 'models/gbt_final.joblib')


NameError: name 'best_gbt' is not defined

In [10]:
# Ensure X, y, X_train, y_train defined (from earlier)
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
import joblib, os

os.makedirs('models', exist_ok=True)

param_grid_gbt = {
    'n_estimators':[200,400],
    'learning_rate':[0.05,0.1],
    'max_depth':[3,4],
    'subsample':[0.8,1.0],
    'max_features':['sqrt', 0.8]
}
gbt = GradientBoostingRegressor(random_state=42)
gcv_gbt = GridSearchCV(gbt, param_grid_gbt, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
gcv_gbt.fit(X_train, y_train)
best_gbt = gcv_gbt.best_estimator_
joblib.dump(best_gbt, 'models/gbt_final.joblib')
print("Saved best_gbt with params:", gcv_gbt.best_params_)


Saved best_gbt with params: {'learning_rate': 0.05, 'max_depth': 4, 'max_features': 'sqrt', 'n_estimators': 200, 'subsample': 0.8}


In [11]:
import numpy as np, pandas as pd, os
from sklearn.metrics import r2_score, mean_absolute_error
import shap, joblib

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_SEED = 42
B_BOOT = 500
B_MODEL_BOOT = 200   # reduce to 200 for speed; increase later if desired
N_SIM = 10000
TAU = 0.7

# 1. Load model if needed
# best_gbt = joblib.load('models/gbt_final.joblib')

# 2. Test predictions for OLS (if available), RF (if available), and GBT
# If you did RF and OLS earlier, ensure y_pred_ols and y_pred_rf exist; otherwise only use GBT
y_pred_gbt = best_gbt.predict(X_test)
gbt_r2 = r2_score(y_test, y_pred_gbt)
gbt_mae = mean_absolute_error(y_test, y_pred_gbt)
print("GBT test R2, MAE:", gbt_r2, gbt_mae)

# 3. Bootstrap test metrics (GB T only if others absent)
rng = np.random.default_rng(RANDOM_SEED)
n_test = len(y_test)
boot_rows = []
for b in range(B_BOOT):
    idx = rng.integers(0, n_test, n_test)
    yb = y_test[idx]
    preds_b = y_pred_gbt[idx]
    boot_rows.append({'b': b, 'gbt_r2': r2_score(yb, preds_b), 'gbt_mae': mean_absolute_error(yb, preds_b)})
pd.DataFrame(boot_rows).to_csv(f'{OUTPUT_DIR}/metrics_bootstrap_draws.csv', index=False)

# 4. SHAP summary
explainer = shap.TreeExplainer(best_gbt)
shap_values = explainer.shap_values(X)            # X is full feature matrix used earlier
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
shap_df = pd.DataFrame({'feature': X.columns, 'mean_abs_shap': mean_abs_shap}).sort_values('mean_abs_shap', ascending=False)
shap_df.to_csv(f'{OUTPUT_DIR}/shap_bootstrap_summary.csv', index=False)

# 5. Monte Carlo baseline sampling
cont = ['env','design','materials','enforcement']
mu = df[cont].mean().values
cov = df[cont].cov().values
Z = rng.multivariate_normal(mu, cov, size=N_SIM)
Z_clipped = np.clip(Z, 0.0, 1.0)
X_sim_cont = pd.DataFrame(Z_clipped, columns=cont)
sample_rows = rng.integers(0, len(df), size=N_SIM)
X_sim_cat = X_cat.iloc[sample_rows].reset_index(drop=True)
X_sim = pd.concat([X_sim_cont.reset_index(drop=True), X_sim_cat.reset_index(drop=True)], axis=1)

def scenario_shift(Xc, scenario):
    Xs = Xc.copy()
    if scenario == 'S_E':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.20, 0, 1)
    if scenario == 'S_M':
        Xs['materials'] = np.clip(Xs['materials'] + 0.20, 0, 1)
    if scenario == 'S_D':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.10, 0, 1)
        Xs['materials'] = np.clip(Xs['materials'] + 0.10, 0, 1)
        Xs['design'] = np.clip(Xs['design'] + 0.05, 0, 1)
    if scenario == 'S_C':
        Xs['enforcement'] = np.clip(Xs['enforcement'] + 0.20, 0, 1)
        Xs['materials'] = np.clip(Xs['materials'] + 0.20, 0, 1)
        Xs['design'] = np.clip(Xs['design'] + 0.10, 0, 1)
    return Xs

scenarios = ['S0','S_E','S_M','S_D','S_C']
scenario_results = []
for s in scenarios:
    Xs = X_sim.copy() if s=='S0' else scenario_shift(X_sim, s)
    preds = best_gbt.predict(Xs)
    scenario_results.append({'scenario': s, 'mu': preds.mean(), 'pi': (preds >= TAU).mean()})
pd.DataFrame(scenario_results).to_csv(f'{OUTPUT_DIR}/scenario_summary.csv', index=False)

# 6. Model-bootstrap for scenarios (reduced B for speed)
scenario_boot = []
for b in range(B_MODEL_BOOT):
    # resample training with replacement
    idxs = rng.integers(0, len(X_train), len(X_train))
    Xb = X_train.iloc[idxs].reset_index(drop=True)
    yb = y_train[idxs]
    model_b = GradientBoostingRegressor(**gcv_gbt.best_params_, random_state=int(RANDOM_SEED+b))
    model_b.fit(Xb, yb)
    for s in scenarios:
        Xs = X_sim.copy() if s=='S0' else scenario_shift(X_sim, s)
        preds_b = model_b.predict(Xs)
        scenario_boot.append({'b': b, 'scenario': s, 'mu': preds_b.mean(), 'pi': (preds_b >= TAU).mean()})
pd.DataFrame(scenario_boot).to_csv(f'{OUTPUT_DIR}/scenario_bootstrap_draws.csv', index=False)

print('Outputs saved to', OUTPUT_DIR)


GBT test R2, MAE: 0.33813164782468685 0.11430307770784795
Outputs saved to outputs
